In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMRegressor, LGBMClassifier
import sys
import re
from sklearn.preprocessing import LabelEncoder
import plotly.express as px
from scipy.stats import spearmanr

In [2]:
df_sold = pd.read_csv("../data/Sold/sold_transactions.csv")
df_sold.head()

C:\Users\mayab\AppData\Local\Temp\ipykernel_6264\4086709524.py:1: DtypeWarning: Columns (0: BuyerAgentAOR, 1: ListAgentAOR, 2: WaterfrontYN, 3: ListAgentEmail, 4: OriginatingSystemName, 5: OriginatingSystemSubName, 6: BuyerAgencyCompensationType, 7: latfilled, 8: lonfilled) have mixed types. Specify dtype option on import or set low_memory=False.
  df_sold = pd.read_csv("../data/Sold/sold_transactions.csv")


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,OriginatingSystemName,OriginatingSystemSubName,BuyerAgencyCompensationType,BuyerAgencyCompensation,latfilled,lonfilled,Year
0,Mlslistings,Mlslistings,"Carpet,Tile,Wood",True,NaN,NaN,False,499000.0,551985747,jwachter@cbnorcal.com,...,6472.0,NaN,NaN,CRMLS,CRMLS_MLSL,NaN,NaN,NaN,NaN,2024
1,HighDesert,HighDesert,NaN,NaN,NaN,NaN,NaN,0.0,535486633,eabrown@lee-associates.com,...,NaN,52320.0,NaN,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN,2024
2,OrangeCounty,OrangeCounty,NaN,True,NaN,NaN,NaN,75000.0,529986282,Joe@9WINWIN.com,...,NaN,217364.0,NaN,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN,2024
3,InlandValleys,InlandValleys,NaN,True,NaN,NaN,NaN,199000.0,529618166,carolthefinder@yahoo.com,...,NaN,217800.0,NaN,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN,2024
4,SouthwestRiversideCounty,SouthwestRiversideCounty,NaN,True,NaN,NaN,NaN,19500.0,522614340,jtavisola@tavisola.com,...,0.0,108883.0,NaN,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN,2024


##### Number of Rows and Columns

In [3]:
df_sold.shape

(615376, 85)

The the sold transactions file has 591115 rows and 87 columns.

##### Review column data types

In [4]:
df_sold.info()

<class 'pandas.DataFrame'>
RangeIndex: 615376 entries, 0 to 615375
Data columns (total 85 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   BuyerAgentAOR                 541725 non-null  str    
 1   ListAgentAOR                  547321 non-null  str    
 2   Flooring                      361626 non-null  str    
 3   ViewYN                        553795 non-null  object 
 4   WaterfrontYN                  338 non-null     object 
 5   BasementYN                    10149 non-null   object 
 6   PoolPrivateYN                 538558 non-null  object 
 7   OriginalListPrice             613592 non-null  float64
 8   ListingKey                    615376 non-null  int64  
 9   ListAgentEmail                571926 non-null  str    
 10  CloseDate                     615376 non-null  str    
 11  ClosePrice                    615369 non-null  float64
 12  ListAgentFirstName            611828 non-null  str    


In [5]:
df_sold.columns

Index(['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'WaterfrontYN',
       'BasementYN', 'PoolPrivateYN', 'OriginalListPrice', 'ListingKey',
       'ListAgentEmail', 'CloseDate', 'ClosePrice', 'ListAgentFirstName',
       'ListAgentLastName', 'Latitude', 'Longitude', 'UnparsedAddress',
       'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket',
       'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName',
       'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName',
       'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName',
       'FireplacesTotal', 'AssociationFeeFrequency', 'AboveGradeFinishedArea',
       'ListingKeyNumeric', 'MLSAreaMajor', 'TaxAnnualAmount',
       'CountyOrParish', 'MlsStatus', 'ElementarySchool', 'AttachedGarageYN',
       'ParkingTotal', 'BuilderName', 'PropertySubType', 'LotSizeAcres',
       'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt', 'StreetNumberNumeric',
       'ListingId', 'BathroomsTotalInteger', 'City', '

In [6]:
df_sold.isna().sum()

BuyerAgentAOR                   73651
ListAgentAOR                    68055
Flooring                       253750
ViewYN                          61581
WaterfrontYN                   615038
                                ...  
BuyerAgencyCompensationType    547629
BuyerAgencyCompensation        547596
latfilled                      519583
lonfilled                      519583
Year                                0
Length: 85, dtype: int64

#### **Missing Value Analysis**

##### Calculate missing counts and percentages per column

In [7]:
missing_sold_counts = df_sold.isnull().sum()
missing_sold_counts

BuyerAgentAOR                   73651
ListAgentAOR                    68055
Flooring                       253750
ViewYN                          61581
WaterfrontYN                   615038
                                ...  
BuyerAgencyCompensationType    547629
BuyerAgencyCompensation        547596
latfilled                      519583
lonfilled                      519583
Year                                0
Length: 85, dtype: int64

In [8]:
missing_sold_percent = (df_sold.isnull().mean()) * 100
missing_sold_percent

BuyerAgentAOR                  11.968455
ListAgentAOR                   11.059092
Flooring                       41.234952
ViewYN                         10.007053
WaterfrontYN                   99.945074
                                 ...    
BuyerAgencyCompensationType    88.990958
BuyerAgencyCompensation        88.985596
latfilled                      84.433420
lonfilled                      84.433420
Year                            0.000000
Length: 85, dtype: float64

In [9]:
missing_summary = pd.DataFrame({
    "missing_sold_counts": missing_sold_counts,
    "missing_sold_percent": missing_sold_percent
})

In [10]:
missing_sold_summary = missing_summary.sort_values(by="missing_sold_percent", ascending=False)
print(missing_sold_summary)

                              missing_sold_counts  missing_sold_percent
MiddleOrJuniorSchoolDistrict               615376                 100.0
ElementarySchoolDistrict                   615376                 100.0
CoveredSpaces                              615376                 100.0
FireplacesTotal                            615376                 100.0
AboveGradeFinishedArea                     615376                 100.0
...                                           ...                   ...
CountyOrParish                                  0                   0.0
ListOfficeName                                  0                   0.0
MlsStatus                                       0                   0.0
StateOrProvince                                 0                   0.0
Year                                            0                   0.0

[85 rows x 2 columns]


In [11]:
missing_sold_summary = missing_sold_summary[missing_sold_summary["missing_sold_counts"] > 0]
print(missing_sold_summary)

                              missing_sold_counts  missing_sold_percent
MiddleOrJuniorSchoolDistrict               615376            100.000000
ElementarySchoolDistrict                   615376            100.000000
CoveredSpaces                              615376            100.000000
FireplacesTotal                            615376            100.000000
AboveGradeFinishedArea                     615376            100.000000
...                                           ...                   ...
ListAgentFullName                             172              0.027950
PostalCode                                    168              0.027300
ListingContractDate                            86              0.013975
ListAgentLastName                              61              0.009913
ClosePrice                                      7              0.001138

[74 rows x 2 columns]


In [12]:
missing_sold_summary["missing_sold_percent"] = missing_sold_summary["missing_sold_percent"].round(2)
print(missing_sold_summary)

                              missing_sold_counts  missing_sold_percent
MiddleOrJuniorSchoolDistrict               615376                100.00
ElementarySchoolDistrict                   615376                100.00
CoveredSpaces                              615376                100.00
FireplacesTotal                            615376                100.00
AboveGradeFinishedArea                     615376                100.00
...                                           ...                   ...
ListAgentFullName                             172                  0.03
PostalCode                                    168                  0.03
ListingContractDate                            86                  0.01
ListAgentLastName                              61                  0.01
ClosePrice                                      7                  0.00

[74 rows x 2 columns]


In [13]:
missing_above_90 = missing_sold_summary[missing_sold_summary['missing_sold_percent'] > 90]
print(missing_above_90)

                              missing_sold_counts  missing_sold_percent
MiddleOrJuniorSchoolDistrict               615376                100.00
ElementarySchoolDistrict                   615376                100.00
CoveredSpaces                              615376                100.00
FireplacesTotal                            615376                100.00
AboveGradeFinishedArea                     615376                100.00
WaterfrontYN                               615038                 99.95
TaxYear                                    614997                 99.94
BusinessType                               613695                 99.73
TaxAnnualAmount                            612791                 99.58
BelowGradeFinishedArea                     612657                 99.56
BasementYN                                 605227                 98.35
BuilderName                                591964                 96.20
LotSizeDimensions                          583070               

In [14]:
missing_above_70 = missing_sold_summary[missing_sold_summary['missing_sold_percent'] > 70]
print(missing_above_70)

                              missing_sold_counts  missing_sold_percent
MiddleOrJuniorSchoolDistrict               615376                100.00
ElementarySchoolDistrict                   615376                100.00
CoveredSpaces                              615376                100.00
FireplacesTotal                            615376                100.00
AboveGradeFinishedArea                     615376                100.00
WaterfrontYN                               615038                 99.95
TaxYear                                    614997                 99.94
BusinessType                               613695                 99.73
TaxAnnualAmount                            612791                 99.58
BelowGradeFinishedArea                     612657                 99.56
BasementYN                                 605227                 98.35
BuilderName                                591964                 96.20
LotSizeDimensions                          583070               

In [15]:
missing_above_70.shape

(28, 2)

In [16]:
missing_above_70

,missing_sold_counts,missing_sold_percent
MiddleOrJuniorSchoolDistrict,615376,100.00
ElementarySchoolDistrict,615376,100.00
CoveredSpaces,615376,100.00
FireplacesTotal,615376,100.00
AboveGradeFinishedArea,615376,100.00
WaterfrontYN,615038,99.95
TaxYear,614997,99.94
BusinessType,613695,99.73
TaxAnnualAmount,612791,99.58
BelowGradeFinishedArea,612657,99.56


Dropping Variables:
- CoveredSpaces                              
- AboveGradeFinishedArea                     
- MiddleOrJuniorSchoolDistrict               
- ElementarySchoolDistrict                   
- FireplacesTotal                            
- WaterfrontYN                               
- TaxYear                                   
- BusinessType                               
- TaxAnnualAmount                            
- BelowGradeFinishedArea                     
- BasementYN                                 
- BuilderName                                
- LotSizeDimensions                          
- CoBuyerAgentFirstName 
- OriginatingSystemName	
- OriginatingSystemSubName	
- ElementarySchool	
- MiddleOrJuniorSchool	
- BuyerAgencyCompensationType
- BuyerAgencyCompensation	
- BuildingAreaTotal	
- HighSchool	
- latfilled	
- lonfilled	
- CoListAgentFirstName	
- CoListAgentLastName	
- CoListOfficeName	
- AssociationFeeFrequency	

To perserve the variables, it would be sufficient if each variable contain less than 70% of missing values in order to use imputation to replace the missingness in the data. 

So, variables with 70% of missing variables will be removed from the dataset because through imputation with variables with a significant number of missingness, it may contribute to bias towards certain values than others. 

In [17]:
df_sold_clean = df_sold.drop(columns=['CoveredSpaces',
'AboveGradeFinishedArea',	
'MiddleOrJuniorSchoolDistrict',
'ElementarySchoolDistrict',
'FireplacesTotal',
'WaterfrontYN',
'TaxYear',
'BusinessType',
'TaxAnnualAmount',	
'BelowGradeFinishedArea',
'BasementYN',	
'BuilderName',	
'LotSizeDimensions',	
'CoBuyerAgentFirstName',	
'OriginatingSystemName',	
'OriginatingSystemSubName',	
'ElementarySchool',	
'MiddleOrJuniorSchool',	
'BuyerAgencyCompensationType',
'BuyerAgencyCompensation',	
'BuildingAreaTotal',	
'HighSchool',	
'latfilled',	
'lonfilled',	
'CoListAgentFirstName',	
'CoListAgentLastName',
'CoListOfficeName',
'AssociationFeeFrequency'	])

In [18]:
df_sold_clean.shape

(615376, 57)

In [19]:
# Calculate percentage for all columns
missing_pct = (df_sold_clean.isna().sum() / len(df_sold_clean)) * 100

# Filter to show only columns that actually have missing data
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

SubdivisionName             64.681756
MainLevelBedrooms           46.923019
Flooring                    41.234952
HighSchoolDistrict          33.387392
AssociationFee              30.966596
AttachedGarageYN            27.743688
Stories                     19.941142
Levels                      13.677491
GarageSpaces                12.785516
PoolPrivateYN               12.483100
BuyerAgentAOR               11.968455
NewConstructionYN           11.872416
ListAgentAOR                11.059092
MLSAreaMajor                11.007904
ViewYN                      10.007053
LotSizeAcres                 9.432282
LotSizeSquareFeet            9.192266
LotSizeArea                  9.115728
FireplaceYN                  8.718085
PropertySubType              7.643457
ListAgentEmail               7.060724
LivingArea                   7.031473
BedroomsTotal                6.632530
BathroomsTotalInteger        4.609215
BuyerOfficeAOR               4.506188
YearBuilt                    4.201821
ParkingTotal

In [20]:
num_cols_with_missing = df_sold_clean.isna().any().sum()

print(f"Number of variables with missing values: {num_cols_with_missing}")

Number of variables with missing values: 46


In [21]:
cols_rows_dropping = missing_pct[(missing_pct < 1) & (missing_pct > 0)]
cols_rows_dropping

OriginalListPrice           0.289904
ClosePrice                  0.001138
ListAgentFirstName          0.576558
ListAgentLastName           0.009913
UnparsedAddress             0.154702
ListPrice                   0.153077
ListAgentFullName           0.027950
BuyerAgentMlsId             0.315092
BuyerAgentFirstName         0.438269
BuyerAgentLastName          0.056226
StreetNumberNumeric         0.249116
City                        0.071664
ContractStatusChangeDate    0.115702
ListingContractDate         0.013975
PostalCode                  0.027300
dtype: float64

In [22]:
cols_rows_dropped = ['OriginalListPrice', 'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'UnparsedAddress', 'ListPrice', 'ListAgentFullName', 'BuyerAgentMlsId',
                'BuyerAgentFirstName', 'BuyerAgentLastName', 'StreetNumberNumeric', 'City', 'ContractStatusChangeDate', 'ListingContractDate', 'PostalCode']

In [23]:
df_sold_clean = df_sold_clean.dropna(subset=['OriginalListPrice', 'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'UnparsedAddress', 'ListPrice', 'ListAgentFullName', 'BuyerAgentMlsId',
                'BuyerAgentFirstName', 'BuyerAgentLastName', 'StreetNumberNumeric', 'City', 'ContractStatusChangeDate', 'ListingContractDate', 'PostalCode'])

In [24]:
df_sold_clean.shape

(603302, 57)

In [25]:
df_sold_clean.shape[0] - df_sold_clean.shape[0]

0

In [26]:
# Calculate percentage for all columns
missing_pct = (df_sold_clean.isna().sum() / len(df_sold_clean)) * 100

# Filter to show only columns that actually have missing data
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

SubdivisionName          64.406384
MainLevelBedrooms        46.149027
Flooring                 41.192139
HighSchoolDistrict       33.404331
AssociationFee           30.424232
AttachedGarageYN         27.718290
Stories                  18.975405
Levels                   12.640601
GarageSpaces             12.623031
BuyerAgentAOR            11.950731
NewConstructionYN        11.658672
PoolPrivateYN            11.519604
ListAgentAOR             11.048198
MLSAreaMajor             10.960514
ViewYN                    9.801559
LotSizeAcres              9.375238
LotSizeSquareFeet         9.133734
LotSizeArea               9.059145
FireplaceYN               8.477512
PropertySubType           7.546304
ListAgentEmail            7.064455
LivingArea                6.818310
BedroomsTotal             6.386685
BathroomsTotalInteger     4.506201
BuyerOfficeAOR            4.382051
YearBuilt                 4.104578
ParkingTotal              2.950927
Latitude                  2.882967
Longitude           

In [27]:
# Create a reference mapping of Names -> Emails
# Drop any rows that have missing emails so the map is 'clean'
email_lookup = df_sold_clean.dropna(subset=['ListAgentEmail']).drop_duplicates(['ListAgentFirstName', 'ListAgentLastName'])

# Set the names as the index to make the 'lookup' possible
email_lookup = email_lookup.set_index(['ListAgentFirstName', 'ListAgentLastName'])['ListAgentEmail']

# Create a function to apply the logic
def fill_missing_emails(row):
    # Only try to fill if the current email is missing
    if pd.isna(row['ListAgentEmail']):
        # Look up the name in our reference map
        return email_lookup.get((row['ListAgentFirstName'], row['ListAgentLastName']), "None")
    return row['ListAgentEmail']

# Apply it to the dataframe
df_sold_clean['ListAgentEmail'] = df_sold_clean.apply(fill_missing_emails, axis=1)

In [28]:
df_sold_clean['ListAgentEmail'].isna().sum()

np.int64(0)

In [29]:
(df_sold_clean['ListAgentEmail'] == 'None').sum()

np.int64(2249)

From the code above, I noticed that there are no missing values for list agent's first and last names, so to determine their emails, I tried to look up any matches by full name to get the list agent's emails for the rows that are missing them. If there are no matches, those emails will be preserved by having a value of "None" (i.e. no email provided).

In [30]:
df_sold_clean.isna().sum()

BuyerAgentAOR                72099
ListAgentAOR                 66654
Flooring                    248513
ViewYN                       59133
PoolPrivateYN                69498
OriginalListPrice                0
ListingKey                       0
ListAgentEmail                   0
CloseDate                        0
ClosePrice                       0
ListAgentFirstName               0
ListAgentLastName                0
Latitude                     17393
Longitude                    17225
UnparsedAddress                  0
PropertyType                     0
LivingArea                   41135
ListPrice                        0
DaysOnMarket                     0
ListOfficeName                   0
BuyerOfficeName               6768
ListAgentFullName                0
BuyerAgentMlsId                  0
BuyerAgentFirstName              0
BuyerAgentLastName               0
ListingKeyNumeric                0
MLSAreaMajor                 66125
CountyOrParish                   0
MlsStatus           

In [31]:
df_sold_clean.columns

Index(['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'PoolPrivateYN',
       'OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'ListAgentFullName', 'BuyerAgentMlsId', 'BuyerAgentFirstName',
       'BuyerAgentLastName', 'ListingKeyNumeric', 'MLSAreaMajor',
       'CountyOrParish', 'MlsStatus', 'AttachedGarageYN', 'ParkingTotal',
       'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR',
       'YearBuilt', 'StreetNumberNumeric', 'ListingId',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal',
       'ContractStatusChangeDate', 'PurchaseContractDate',
       'ListingContractDate', 'StateOrProvince', 'FireplaceYN', 'Stories',
       'Levels', 'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN',
       'GarageSpaces

#### **Convert date fields to datetime format (CloseDate, PurchaseContractDate, ListingContractDate, ContractStatusChangeDate)**

In [32]:
df_sold_clean['CloseDate'] = pd.to_datetime(df_sold_clean['CloseDate'], errors='coerce')

In [33]:
df_sold_clean['PurchaseContractDate'] = pd.to_datetime(df_sold_clean['PurchaseContractDate'], errors='coerce')

In [34]:
df_sold_clean['ListingContractDate'] = pd.to_datetime(df_sold_clean['ListingContractDate'], errors='coerce')

In [35]:
df_sold_clean['ContractStatusChangeDate'] = pd.to_datetime(df_sold_clean['ContractStatusChangeDate'], errors='coerce')

In [36]:
print(df_sold_clean['PurchaseContractDate'].dtype)

datetime64[us]


#### **Remove unnecessary or redundant columns**

In [37]:
df_sold_clean.columns

Index(['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'PoolPrivateYN',
       'OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'ListAgentFullName', 'BuyerAgentMlsId', 'BuyerAgentFirstName',
       'BuyerAgentLastName', 'ListingKeyNumeric', 'MLSAreaMajor',
       'CountyOrParish', 'MlsStatus', 'AttachedGarageYN', 'ParkingTotal',
       'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR',
       'YearBuilt', 'StreetNumberNumeric', 'ListingId',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal',
       'ContractStatusChangeDate', 'PurchaseContractDate',
       'ListingContractDate', 'StateOrProvince', 'FireplaceYN', 'Stories',
       'Levels', 'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN',
       'GarageSpaces

In [38]:
df_sold_clean.shape

(603302, 57)

In [39]:
(df_sold_clean['OriginalListPrice'] == df_sold_clean['ListPrice']).value_counts()

True     410760
False    192542
Name: count, dtype: int64

No redundant columns

#### **Outlier Cleaning**

Remove or flag invalid numeric values: ClosePrice <=0, LivingArea <=0, DaysOnMarket < 0, negative Bedrooms or Bathrooms

Removing and inspecting negative values

In [40]:
neg_counts = (df_sold_clean.select_dtypes(include='number') < 0).sum()

In [41]:
neg_counts[neg_counts > 0]

Latitude             5
Longitude       585969
DaysOnMarket        60
ParkingTotal        95
dtype: int64

In [42]:
(neg_counts[neg_counts > 0] / df_sold_clean.shape[0] * 100).sort_values(ascending=False)

Longitude       97.126978
ParkingTotal     0.015747
DaysOnMarket     0.009945
Latitude         0.000829
dtype: float64

ParkingTotal

In [43]:
neg_ParkingTotal = df_sold_clean[df_sold_clean['ParkingTotal'] < 0]
neg_ParkingTotal.head()

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,Levels,LotSizeArea,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,Year
6424,Mlslistings,Mlslistings,Wood,True,False,639000.0,1050719913,josie.realtimerealty@gmail.com,2024-01-19,650000.0,...,NaN,5147.00,NaN,False,2.0,North Monterey County Unified,95012,NaN,5147.00,2024
9489,Mlslistings,Mlslistings,NaN,False,False,899999.0,1047166342,ascenciorealestate@gmail.com,2024-01-03,905000.0,...,NaN,9583.00,NaN,False,0.0,San Jose Unified,95110,NaN,9583.00,2024
14274,Mlslistings,Mlslistings,"Laminate,Tile,Wood",False,False,1280000.0,1045843310,marytan@compass.com,2024-01-08,1260000.0,...,NaN,938.00,NaN,False,0.0,Other,95014,448.0,938.00,2024
14468,Mlslistings,Mlslistings,Laminate,True,NaN,850000.0,1045777028,carmelasoto415@gmail.com,2024-01-23,770000.0,...,NaN,301818.53,NaN,False,1.0,South San Francisco Unified,94080,722.0,301818.53,2024
15725,Mlslistings,Mlslistings,"Laminate,Tile",False,NaN,779000.0,1044839849,nitortiz@gmail.com,2024-01-08,755000.0,...,NaN,6098.00,NaN,NaN,0.0,Sacramento City Unified,95822,NaN,6098.00,2024


In [44]:
neg_ParkingTotal.head()

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,Levels,LotSizeArea,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,Year
6424,Mlslistings,Mlslistings,Wood,True,False,639000.0,1050719913,josie.realtimerealty@gmail.com,2024-01-19,650000.0,...,NaN,5147.00,NaN,False,2.0,North Monterey County Unified,95012,NaN,5147.00,2024
9489,Mlslistings,Mlslistings,NaN,False,False,899999.0,1047166342,ascenciorealestate@gmail.com,2024-01-03,905000.0,...,NaN,9583.00,NaN,False,0.0,San Jose Unified,95110,NaN,9583.00,2024
14274,Mlslistings,Mlslistings,"Laminate,Tile,Wood",False,False,1280000.0,1045843310,marytan@compass.com,2024-01-08,1260000.0,...,NaN,938.00,NaN,False,0.0,Other,95014,448.0,938.00,2024
14468,Mlslistings,Mlslistings,Laminate,True,NaN,850000.0,1045777028,carmelasoto415@gmail.com,2024-01-23,770000.0,...,NaN,301818.53,NaN,False,1.0,South San Francisco Unified,94080,722.0,301818.53,2024
15725,Mlslistings,Mlslistings,"Laminate,Tile",False,NaN,779000.0,1044839849,nitortiz@gmail.com,2024-01-08,755000.0,...,NaN,6098.00,NaN,NaN,0.0,Sacramento City Unified,95822,NaN,6098.00,2024


In [45]:
df_sold_clean['ParkingTotal'].describe()

count    5.854990e+05
mean     7.496302e+00
std      3.400497e+03
min     -1.430000e+02
25%      2.000000e+00
50%      2.000000e+00
75%      3.000000e+00
max      2.593669e+06
Name: ParkingTotal, dtype: float64

DaysOnMarket

In [46]:
neg_DaysOnMarket = df_sold_clean[df_sold_clean['DaysOnMarket'] < 0]
neg_DaysOnMarket.head()

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,Levels,LotSizeArea,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,Year
31,CoastalMendocino,CoastalMendocino,Tile,True,False,1950000.0,1063453216,kira@mendosir.com,2024-01-22,1950000.0,...,Two,1.71,NaN,NaN,2.0,NaN,95437,NaN,74487.6,2024
826,VenturaCoastal,VenturaCoastal,Vinyl,False,True,2975.0,1055498337,cindyisalways@gmail.com,2024-01-17,2975.0,...,One,3920.00,NaN,NaN,2.0,NaN,93041,0.0,3920.0,2024
1676,PasadenaFoothills,PasadenaFoothills,Laminate,False,False,4000.0,1054112664,arbi.derian@compass.com,2024-01-16,4000.0,...,ThreeOrMore,6098.00,0.0,False,2.0,NaN,91601,0.0,6098.0,2024
3838,PasadenaFoothills,PasadenaFoothills,Vinyl,False,False,3950.0,1052391149,amyponce888@gmail.com,2024-01-31,3950.0,...,One,5661.00,NaN,False,0.0,NaN,90041,0.0,5661.0,2024
9426,CaliforniaDesert,CaliforniaDesert,Carpet,True,False,120000.0,1047183932,martyjelmberg@gmail.com,2024-01-12,120000.0,...,One,800.00,NaN,False,NaN,NaN,92241,NaN,800.0,2024


In [47]:
rows_DaysOnMarket = neg_DaysOnMarket['DaysOnMarket']
rows_DaysOnMarket.head()

31     -36
826    -11
1676    -1
3838    -1
9426    -4
Name: DaysOnMarket, dtype: int64

In [48]:
non_neg = (df_sold_clean['DaysOnMarket'] >= 0) & (df_sold_clean['ParkingTotal'] >= 0)

In [49]:
df_sold_clean = df_sold_clean[non_neg].copy()

In [50]:
df_sold_clean.shape

(585346, 57)

In [51]:
neg_counts = (df_sold_clean.select_dtypes(include='number') < 0).sum()

In [52]:
neg_counts[neg_counts > 0]

Latitude          3
Longitude    568062
dtype: int64

In [53]:
df_sold_clean['missing_coordinates_flag'] = df_sold_clean['Latitude'].isna() | df_sold_clean['Longitude'].isna()

In [54]:
df_sold_clean['missing_coordinates_flag'].value_counts()

missing_coordinates_flag
False    567986
True      17360
Name: count, dtype: int64

In [55]:
df_sold_clean['sentinel_null_flag'] = (df_sold_clean['Latitude'] == 0) | (df_sold_clean['Longitude'] == 0)

In [56]:
df_sold_clean['sentinel_null_flag'].value_counts()

sentinel_null_flag
False    585307
True         39
Name: count, dtype: int64

In [57]:
df_sold_clean['positive_longitude_flag'] = df_sold_clean['Longitude'] > 0

In [58]:
df_sold_clean['positive_longitude_flag'].value_counts()

positive_longitude_flag
False    585293
True         53
Name: count, dtype: int64

In [59]:
df_sold_clean['out_of_state_flag'] = (
    ~(
        (df_sold_clean['Latitude'] >= 32.5) &
        (df_sold_clean['Latitude'] <= 42.0) &
        (df_sold_clean['Longitude'] >= -124.5) &
        (df_sold_clean['Longitude'] <= -114.1)
    )
    & ~df_sold_clean['missing_coordinates_flag']
    & ~df_sold_clean['sentinel_null_flag']
)

In [60]:
df_sold_clean['out_of_state_flag'].value_counts()

out_of_state_flag
False    585235
True        111
Name: count, dtype: int64

In [62]:
df_sold_clean.shape

(585346, 61)

In [63]:
'''
df_map = df_sold_clean.sample(30000)

fig = px.scatter_mapbox(
    df_map,
    lat="Latitude",
    lon="Longitude",
    color="OriginalListPrice",
    size_max=15,
    zoom=5,
    title="California Property Value Distribution",
    color_continuous_scale=px.colors.sequential.Plasma,
    mapbox_style="open-street-map"
)

fig.show(renderer='browser')
'''

'\ndf_map = df_sold_clean.sample(30000)\n\nfig = px.scatter_mapbox(\n    df_map,\n    lat="Latitude",\n    lon="Longitude",\n    color="OriginalListPrice",\n    size_max=15,\n    zoom=5,\n    title="California Property Value Distribution",\n    color_continuous_scale=px.colors.sequential.Plasma,\n    mapbox_style="open-street-map"\n)\n\nfig.show(renderer=\'browser\')\n'

#### **Further Handling Missing Values Appropriately**

In [64]:
missing_sold_counts = df_sold_clean.isnull().sum()
missing_sold_counts

BuyerAgentAOR                69726
ListAgentAOR                 64282
Flooring                    230662
ViewYN                       55605
PoolPrivateYN                51623
                             ...  
Year                             0
missing_coordinates_flag         0
sentinel_null_flag               0
positive_longitude_flag          0
out_of_state_flag                0
Length: 61, dtype: int64

In [65]:
missing_sold_percent = (df_sold_clean.isnull().mean()) * 100
missing_sold_percent

BuyerAgentAOR               11.911929
ListAgentAOR                10.981881
Flooring                    39.406095
ViewYN                       9.499510
PoolPrivateYN                8.819228
                              ...    
Year                         0.000000
missing_coordinates_flag     0.000000
sentinel_null_flag           0.000000
positive_longitude_flag      0.000000
out_of_state_flag            0.000000
Length: 61, dtype: float64

In [66]:
missing_summary = pd.DataFrame({
    "missing_sold_counts": missing_sold_counts,
    "missing_sold_percent": missing_sold_percent
})

In [67]:
missing_sold_summary = missing_summary.sort_values(by="missing_sold_percent", ascending=False)
print(missing_sold_summary)

                          missing_sold_counts  missing_sold_percent
SubdivisionName                        372443             63.627837
MainLevelBedrooms                      260505             44.504447
Flooring                               230662             39.406095
HighSchoolDistrict                     183709             31.384685
AssociationFee                         179463             30.659302
...                                       ...                   ...
Year                                        0              0.000000
missing_coordinates_flag                    0              0.000000
sentinel_null_flag                          0              0.000000
positive_longitude_flag                     0              0.000000
out_of_state_flag                           0              0.000000

[61 rows x 2 columns]


In [68]:
missing_sold_summary[missing_sold_summary['missing_sold_percent'] > 0]

,missing_sold_counts,missing_sold_percent
SubdivisionName,372443,63.627837
MainLevelBedrooms,260505,44.504447
Flooring,230662,39.406095
HighSchoolDistrict,183709,31.384685
AssociationFee,179463,30.659302
AttachedGarageYN,149406,25.524391
Stories,96585,16.500497
BuyerAgentAOR,69726,11.911929
NewConstructionYN,66813,11.414275
ListAgentAOR,64282,10.981881


- Under 10% missing -> simple statistical imputation method, but for categorical variables, may include a "Missing" category if deemed necessary
- 10-50% missing -> "Decision Zone" - meaning that there are different approaches that can be taken to impute the missing values like a group-based imputation, model-based if justifiable, adding a missing indicator, or doing simple imputation if necessary
- above 50% -> Since at this point, they are probably important features, the missingness in the data could be important and signals themselves. 

Let's start with "Under 10% missing"

In [69]:
under_10 = missing_sold_summary[(missing_sold_summary['missing_sold_percent'] < 10) & (missing_sold_percent != 0)]

C:\Users\mayab\AppData\Local\Temp\ipykernel_6264\637377579.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  under_10 = missing_sold_summary[(missing_sold_summary['missing_sold_percent'] < 10) & (missing_sold_percent != 0)]


In [70]:
under_10

,missing_sold_counts,missing_sold_percent
Levels,58367,9.971367
GarageSpaces,58354,9.969146
LotSizeAcres,56367,9.629689
ViewYN,55605,9.499510
LotSizeSquareFeet,54906,9.380093
LotSizeArea,54459,9.303728
PoolPrivateYN,51623,8.819228
FireplaceYN,33334,5.694751
PropertySubType,31060,5.306263
BuyerOfficeAOR,26374,4.505711


In [71]:
df_sold_clean['GarageSpaces'].value_counts().head()

GarageSpaces
2.0    310053
0.0     84175
1.0     65172
3.0     55478
4.0      7584
Name: count, dtype: int64

In [72]:
df_sold_clean['GarageSpaces'].value_counts(normalize=True).head()

GarageSpaces
2.0    0.588345
0.0    0.159727
1.0    0.123668
3.0    0.105273
4.0    0.014391
Name: proportion, dtype: float64

In [73]:
df_sold_clean['GarageSpaces_Missing'] = df_sold_clean['GarageSpaces'].isna().astype(int)

In [74]:
df_sold_clean['GarageSpaces'] = df_sold_clean['GarageSpaces'].fillna(df_sold_clean['GarageSpaces'].median())

In [75]:
df_sold_clean['GarageSpaces'].value_counts().head()

GarageSpaces
2.0    368407
0.0     84175
1.0     65172
3.0     55478
4.0      7584
Name: count, dtype: int64

In [76]:
df_sold_clean['GarageSpaces'].value_counts(normalize=True).head()

GarageSpaces
2.0    0.629383
0.0    0.143804
1.0    0.111339
3.0    0.094778
4.0    0.012956
Name: proportion, dtype: float64

In [77]:
df_sold_clean['LotSizeAcres'].describe()

count    5.289790e+05
mean     7.706308e+02
std      5.092735e+05
min      0.000000e+00
25%      1.182000e-01
50%      1.653000e-01
75%      2.758500e-01
max      3.702600e+08
Name: LotSizeAcres, dtype: float64

In [78]:
df_sold_clean['LotSizeAcres'] = df_sold_clean['LotSizeAcres'].fillna(df_sold_clean['LotSizeAcres'].median())

In [79]:
df_sold_clean['LotSizeAcres'].isna().sum()

np.int64(0)

The mean is greater than the median. The values are skewed to the right. I will be using the median to impute.

In [80]:
df_sold_clean['ViewYN'].value_counts()

ViewYN
True     320172
False    209569
Name: count, dtype: int64

In [81]:
df_sold_clean['ViewYN'].value_counts(normalize=True)

ViewYN
True     0.604393
False    0.395607
Name: proportion, dtype: float64

In [82]:
df_sold_clean['ViewYN'] = df_sold_clean['ViewYN'].fillna(df_sold_clean['ViewYN'].mode()[0])

In [83]:
df_sold_clean['ViewYN'].value_counts(normalize=True)

ViewYN
True     0.641974
False    0.358026
Name: proportion, dtype: float64

In [84]:
df_sold_clean['LotSizeSquareFeet'].describe()

count    5.304400e+05
mean     4.563517e+05
std      2.034074e+07
min      0.000000e+00
25%      5.113000e+03
50%      7.200000e+03
75%      1.200000e+04
max      5.193920e+09
Name: LotSizeSquareFeet, dtype: float64

The mean is less than the median. The values are skewed to the left. I will be using the median to impute the missing values. 

In [85]:
df_sold_clean['LotSizeSquareFeet'] = df_sold_clean['LotSizeSquareFeet'].fillna(df_sold_clean['LotSizeSquareFeet'].median())

In [86]:
df_sold_clean['LotSizeSquareFeet'].isna().sum()

np.int64(0)

In [87]:
df_sold_clean['LotSizeArea'].describe()

count    5.308870e+05
mean     4.476724e+04
std      2.327891e+06
min      0.000000e+00
25%      4.928000e+03
50%      7.000000e+03
75%      1.100100e+04
max      9.187423e+08
Name: LotSizeArea, dtype: float64

The mean is less than the median. The values are skewed to the left. I will be using the median to impute.

In [88]:
df_sold_clean['LotSizeArea'] = df_sold_clean['LotSizeArea'].fillna(df_sold_clean['LotSizeArea'].median())

In [89]:
df_sold_clean['LotSizeArea'].isna().sum()

np.int64(0)

In [90]:
df_sold_clean['PoolPrivateYN'].value_counts()

PoolPrivateYN
False    470902
True      62821
Name: count, dtype: int64

In [91]:
df_sold_clean['PoolPrivateYN'].value_counts(normalize=True)

PoolPrivateYN
False    0.882297
True     0.117703
Name: proportion, dtype: float64

In [92]:
df_sold_clean['PoolPrivateYN'] = df_sold_clean['PoolPrivateYN'].fillna(df_sold_clean['PoolPrivateYN'].mode()[0])

In [93]:
df_sold_clean['PoolPrivateYN'].value_counts(normalize=True)

PoolPrivateYN
False    0.892677
True     0.107323
Name: proportion, dtype: float64

In [94]:
df_sold_clean['FireplaceYN'].value_counts()

FireplaceYN
True     338665
False    213347
Name: count, dtype: int64

In [95]:
df_sold_clean['FireplaceYN'].value_counts(normalize=True)

FireplaceYN
True     0.61351
False    0.38649
Name: proportion, dtype: float64

In [96]:
df_sold_clean['FireplaceYN'] = df_sold_clean['FireplaceYN'].fillna(df_sold_clean['FireplaceYN'].mode()[0])

In [97]:
df_sold_clean['FireplaceYN'].value_counts(normalize=True)

FireplaceYN
True     0.63552
False    0.36448
Name: proportion, dtype: float64

In [98]:
df_sold_clean['PropertySubType'].value_counts().head()

PropertySubType
SingleFamilyResidence    369128
Condominium              105578
Townhouse                 34839
Apartment                 13859
Duplex                    11476
Name: count, dtype: int64

In [99]:
df_sold_clean['PropertySubType'].value_counts(normalize=True).head()

PropertySubType
SingleFamilyResidence    0.665952
Condominium              0.190476
Townhouse                0.062854
Apartment                0.025003
Duplex                   0.020704
Name: proportion, dtype: float64

In [100]:
df_sold_clean['PropertySubType'] = df_sold_clean['PropertySubType'].fillna(df_sold_clean['PropertySubType'].mode()[0])

In [101]:
df_sold_clean['PropertySubType'].value_counts(normalize=True).head()

PropertySubType
SingleFamilyResidence    0.683678
Condominium              0.180369
Townhouse                0.059519
Apartment                0.023677
Duplex                   0.019605
Name: proportion, dtype: float64

In [102]:
df_sold_clean['BuyerOfficeAOR'].value_counts(normalize=True).head()

BuyerOfficeAOR
OrangeCounty    0.097479
PacificWest     0.068841
SanDiego        0.066273
Southland       0.059405
Mlslistings     0.046194
Name: proportion, dtype: float64

In [103]:
df_sold_clean['BuyerOfficeAOR'] = df_sold_clean['BuyerOfficeAOR'].fillna("Missing")

In [104]:
df_sold_clean['BuyerOfficeAOR'].value_counts(normalize=True).head()

BuyerOfficeAOR
OrangeCounty    0.093087
PacificWest     0.065739
SanDiego        0.063287
Southland       0.056729
Missing         0.045057
Name: proportion, dtype: float64

In [105]:
df_sold_clean['LivingArea'].describe()

count    5.620140e+05
mean     3.457925e+03
std      1.212856e+06
min      0.000000e+00
25%      1.183000e+03
50%      1.578000e+03
75%      2.149000e+03
max      9.090909e+08
Name: LivingArea, dtype: float64

The mean is greater than the median, which shows that it is skewing to the right. I will then impute by the median rather than the mean.

In [106]:
df_sold_clean['LivingArea'] = df_sold_clean['LivingArea'].fillna(df_sold_clean['LivingArea'].median())

In [107]:
df_sold_clean['LivingArea'].describe()

count    5.853460e+05
mean     3.382991e+03
std      1.188438e+06
min      0.000000e+00
25%      1.200000e+03
50%      1.578000e+03
75%      2.114000e+03
max      9.090909e+08
Name: LivingArea, dtype: float64

In [108]:
df_sold_clean['LivingArea'].isna().sum()

np.int64(0)

In [109]:
df_sold_clean['BedroomsTotal'].value_counts().head()

BedroomsTotal
3.0    210898
4.0    133457
2.0    133013
5.0     38836
1.0     33273
Name: count, dtype: int64

In [110]:
df_sold_clean['BedroomsTotal'].value_counts(normalize=True).head()

BedroomsTotal
3.0    0.373520
4.0    0.236365
2.0    0.235578
5.0    0.068782
1.0    0.058930
Name: proportion, dtype: float64

In [111]:
df_sold_clean['BedroomsTotal_Missing'] = df_sold_clean['BedroomsTotal'].isna().astype(int)

In [112]:
df_sold_clean['BedroomsTotal'] = df_sold_clean['BedroomsTotal'].fillna(df_sold_clean['BedroomsTotal'].median())

In [113]:
df_sold_clean['BedroomsTotal'].isna().sum()

np.int64(0)

In [114]:
df_sold_clean['Latitude'].describe()

count    567986.000000
mean         34.483034
std           1.571086
min        -117.472493
25%          33.732635
50%          34.027222
75%          34.263709
max         157.000000
Name: Latitude, dtype: float64

The median and the mean are about the same measure, so I will use the mean to impute missing values.

In [115]:
df_sold_clean['Latitude'] = df_sold_clean['Latitude'].fillna(df_sold_clean['Latitude'].mean())

In [116]:
df_sold_clean['Latitude'].isna().sum()

np.int64(0)

In [117]:
df_sold_clean['Longitude'].describe()

count    568154.000000
mean       -118.429333
std           2.941356
min        -177.646696
25%        -118.515149
50%        -118.039264
75%        -117.331829
max         329.000000
Name: Longitude, dtype: float64

The median and the mean are about the same measure, so I will use the mean to impute missing values.

In [118]:
df_sold_clean['Longitude'] = df_sold_clean['Longitude'].fillna(df_sold_clean['Longitude'].mean())

In [119]:
df_sold_clean['Longitude'].isna().sum()

np.int64(0)

In [120]:
df_sold_clean['PurchaseContractDate'].isna().sum()

np.int64(12200)

In [121]:
df_sold_clean['No_Purchase_Date'] = df_sold_clean['PurchaseContractDate'].isna()

In [122]:
df_sold_clean['YearBuilt'].describe()

count    575728.000000
mean       1978.208988
std          27.070946
min        1776.000000
25%        1960.000000
50%        1979.000000
75%        1999.000000
max        2026.000000
Name: YearBuilt, dtype: float64

The mean and the median are about the same

In [123]:
df_sold_clean['YearBuilt'] = df_sold_clean['YearBuilt'].fillna(round(df_sold_clean['YearBuilt'].mean()))

In [124]:
(df_sold_clean['YearBuilt'] % 1 == 0).value_counts()

YearBuilt
True    585346
Name: count, dtype: int64

In [125]:
df_sold_clean['BathroomsTotalInteger'].describe()

count    575963.000000
mean          2.450039
std           1.424797
min           0.000000
25%           2.000000
50%           2.000000
75%           3.000000
max         175.000000
Name: BathroomsTotalInteger, dtype: float64

In [126]:
df_sold_clean['BathroomsTotalInteger'].value_counts(normalize=True).head()

BathroomsTotalInteger
2.0    0.416107
3.0    0.308350
1.0    0.140545
4.0    0.070383
5.0    0.025505
Name: proportion, dtype: float64

In [127]:
df_sold_clean['BathroomsTotalInteger'] = df_sold_clean['BathroomsTotalInteger'].fillna(round(df_sold_clean['BathroomsTotalInteger'].mean()))

In [128]:
df_sold_clean['BathroomsTotalInteger'].value_counts(normalize=True).head()

BathroomsTotalInteger
2.0    0.425466
3.0    0.303407
1.0    0.138293
4.0    0.069255
5.0    0.025096
Name: proportion, dtype: float64

In [129]:
df_sold_clean['BuyerOfficeName'].value_counts(normalize=True).head()

BuyerOfficeName
Compass                   0.062237
Coldwell Banker Realty    0.037350
NONMEMBER MRML            0.020007
None MRML                 0.014381
Keller Williams Realty    0.014341
Name: proportion, dtype: float64

In [130]:
df_sold_clean['BuyerOfficeName'] = df_sold_clean['BuyerOfficeName'].fillna("Missing")

In [131]:
df_sold_clean['BuyerOfficeName'].value_counts(normalize=True).head()

BuyerOfficeName
Compass                   0.061521
Coldwell Banker Realty    0.036920
NONMEMBER MRML            0.019776
None MRML                 0.014216
Keller Williams Realty    0.014176
Name: proportion, dtype: float64

In [132]:
df_sold_clean['BuyerOfficeName'].isna().sum()

np.int64(0)

Onto the 10-50% missing

In [133]:
btwn_10_50 = missing_sold_summary[(missing_sold_summary['missing_sold_percent'] >= 10) & (missing_sold_percent <= 50)]

C:\Users\mayab\AppData\Local\Temp\ipykernel_6264\1507415290.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  btwn_10_50 = missing_sold_summary[(missing_sold_summary['missing_sold_percent'] >= 10) & (missing_sold_percent <= 50)]


In [134]:
btwn_10_50

,missing_sold_counts,missing_sold_percent
MainLevelBedrooms,260505,44.504447
Flooring,230662,39.406095
HighSchoolDistrict,183709,31.384685
AssociationFee,179463,30.659302
AttachedGarageYN,149406,25.524391
Stories,96585,16.500497
BuyerAgentAOR,69726,11.911929
NewConstructionYN,66813,11.414275
ListAgentAOR,64282,10.981881
MLSAreaMajor,63628,10.870152


In [135]:
df_sold_clean['MainLevelBedrooms'].describe()

count    324841.000000
mean          1.931268
std           6.148786
min           0.000000
25%           1.000000
50%           2.000000
75%           3.000000
max        3333.000000
Name: MainLevelBedrooms, dtype: float64

In [136]:
df_sold_clean['MainLevelBedrooms'].value_counts(normalize=True).head()

MainLevelBedrooms
3.0    0.251677
1.0    0.249125
2.0    0.190635
0.0    0.183437
4.0    0.107000
Name: proportion, dtype: float64

In [137]:
df_sold_clean['MainLevelBedrooms_Missing'] = df_sold_clean['MainLevelBedrooms'].isna().astype(int)

In [138]:
df_sold_clean['MainLevelBedrooms'] = df_sold_clean['MainLevelBedrooms'].fillna(df_sold_clean['MainLevelBedrooms'].median())

In [139]:
df_sold_clean['MainLevelBedrooms'].value_counts(normalize=True).head()

MainLevelBedrooms
2.0    0.550838
3.0    0.139670
1.0    0.138253
0.0    0.101800
4.0    0.059380
Name: proportion, dtype: float64

The mean and the median is about the same, so using mean to impute the missing values but also add a missing indicator.

In [140]:
df_sold_clean['MainLevelBedrooms'].isna().sum()

np.int64(0)

In [141]:
df_sold_clean['Flooring'].value_counts(normalize=True).head()

Flooring
Wood           0.112148
Laminate       0.109441
Carpet,Tile    0.098880
Tile,Wood      0.076665
Vinyl          0.055153
Name: proportion, dtype: float64

In [142]:
df_sold_clean['Flooring'] = df_sold_clean['Flooring'].fillna("Missing")

In [143]:
df_sold_clean['Flooring'].value_counts().head()

Flooring
Missing        230662
Wood            39777
Laminate        38817
Carpet,Tile     35071
Tile,Wood       27192
Name: count, dtype: int64

In [144]:
df_sold_clean['Flooring'].isna().sum()

np.int64(0)

In [145]:
df_sold_clean['HighSchoolDistrict'].value_counts().head()

HighSchoolDistrict
Los Angeles Unified          38754
Other                        33060
Capistrano Unified           15035
Irvine Unified               10128
Saddleback Valley Unified     8838
Name: count, dtype: int64

In [146]:
df_sold_clean['HighSchoolDistrict'] = df_sold_clean['HighSchoolDistrict'].fillna("Missing")

In [147]:
df_sold_clean['HighSchoolDistrict'].value_counts().head()

HighSchoolDistrict
Missing                183709
Los Angeles Unified     38754
Other                   33060
Capistrano Unified      15035
Irvine Unified          10128
Name: count, dtype: int64

In [148]:
df_sold_clean['AssociationFee'].describe()

count    405883.000000
mean        189.925035
std        1637.736851
min           0.000000
25%           0.000000
50%           0.000000
75%         300.000000
max      750000.000000
Name: AssociationFee, dtype: float64

In [149]:
# Regression Predictions
def impute_lgbm_numeric(df, target):
    df = df.copy()

    # mask missing values
    missing_mask = df[target].isna()

    if missing_mask.sum() == 0:
        return df

    # split data
    train = df[~missing_mask]
    test = df[missing_mask]

    y_train = train[target]
    X_train = train.drop(columns=[target]).select_dtypes(include='number')
    X_test = test.drop(columns=[target]).select_dtypes(include='number')

    # encode categorical columns safely
    for col in X_train.select_dtypes(include=['object', 'category']).columns:
        X_train[col] = X_train[col].astype('category').cat.codes
        X_test[col] = X_test[col].astype('category').cat.codes

    # model
    model = LGBMRegressor(
        n_estimators=200,
        learning_rate=0.05,
        random_state=42,
        verbosity=-1
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    # constraint: association_fee cannot be negative
    preds = np.clip(preds, a_min=0, a_max=None)

    df.loc[missing_mask, target] = preds

    return df

In [150]:
df_sold_clean = impute_lgbm_numeric(df_sold_clean, "AssociationFee")

In [151]:
df_sold_clean['AssociationFee'].describe()

count    585346.000000
mean        203.343913
std        1402.747027
min           0.000000
25%           0.000000
50%          85.682814
75%         277.000000
max      750000.000000
Name: AssociationFee, dtype: float64

In [152]:
df_sold_clean['AssociationFee'].isna().sum()

np.int64(0)

In [153]:
(df_sold_clean['AssociationFee'] < 0).value_counts()

AssociationFee
False    585346
Name: count, dtype: int64

In [154]:
df_sold_clean['AttachedGarageYN'].value_counts()

AttachedGarageYN
True     339934
False     96006
Name: count, dtype: int64

In [155]:
df_sold_clean['AttachedGarageYN'] = df_sold_clean['AttachedGarageYN'].fillna("Missing")

In [156]:
df_sold_clean['AttachedGarageYN'].value_counts()

AttachedGarageYN
True       339934
Missing    149406
False       96006
Name: count, dtype: int64

In [157]:
df_sold_clean['Stories'].describe()

count    488761.000000
mean          1.364722
std           0.481353
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max           2.000000
Name: Stories, dtype: float64

In [158]:
df_sold_clean['Stories'].value_counts()

Stories
1.0    310499
2.0    178262
Name: count, dtype: int64

In [159]:
df_sold_clean['Stories'].value_counts(normalize=True)

Stories
1.0    0.635278
2.0    0.364722
Name: proportion, dtype: float64

In [160]:
df_sold_clean['Stories_Missing'] = df_sold_clean['Stories'].isna().astype(int)

In [161]:
df_sold_clean['Stories'] = df_sold_clean['Stories'].fillna(df_sold_clean['Stories'].median())

In [162]:
df_sold_clean['Stories'].value_counts()

Stories
1.0    407084
2.0    178262
Name: count, dtype: int64

In [163]:
df_sold_clean['Stories'].value_counts(normalize=True)

Stories
1.0    0.695459
2.0    0.304541
Name: proportion, dtype: float64

In [164]:
df_sold_clean['BuyerAgentAOR'].value_counts().head()

BuyerAgentAOR
OrangeCounty             51774
BeverlyHillsGreaterLa    35808
PacificWest              34946
SanDiego                 33053
Southland                32551
Name: count, dtype: int64

In [165]:
df_sold_clean['BuyerAgentAOR'] = df_sold_clean['BuyerAgentAOR'].fillna("Missing")

In [166]:
df_sold_clean['BuyerAgentAOR'].value_counts().head()

BuyerAgentAOR
Missing                  69726
OrangeCounty             51774
BeverlyHillsGreaterLa    35808
PacificWest              34946
SanDiego                 33053
Name: count, dtype: int64

In [167]:
df_sold_clean['ListAgentAOR'].value_counts().head()

ListAgentAOR
OrangeCounty             52480
BeverlyHillsGreaterLa    35942
PacificWest              35268
Southland                33551
SanDiego                 33184
Name: count, dtype: int64

In [168]:
df_sold_clean['ListAgentAOR'] = df_sold_clean['ListAgentAOR'].fillna("Missing")

In [169]:
df_sold_clean['ListAgentAOR'].value_counts().head()

ListAgentAOR
Missing                  64282
OrangeCounty             52480
BeverlyHillsGreaterLa    35942
PacificWest              35268
Southland                33551
Name: count, dtype: int64

In [170]:
df_sold_clean['NewConstructionYN'].value_counts()

NewConstructionYN
False    497176
True      21357
Name: count, dtype: int64

In [171]:
df_sold_clean['NewConstructionYN'].value_counts(normalize=True)

NewConstructionYN
False    0.958813
True     0.041187
Name: proportion, dtype: float64

In [172]:
df_sold_clean['NewConstructionYN'] = df_sold_clean['NewConstructionYN'].fillna(df_sold_clean['NewConstructionYN'].mode()[0])

In [173]:
df_sold_clean['NewConstructionYN'].value_counts(normalize=True)

NewConstructionYN
False    0.963514
True     0.036486
Name: proportion, dtype: float64

In [174]:
df_sold_clean['MLSAreaMajor'].value_counts().head()

MLSAreaMajor
699 - Not Defined                     48813
SRCAR - Southwest Riverside County    24930
252 - Riverside                        6832
248 - Corona                           4422
686 - Ontario                          3903
Name: count, dtype: int64

In [175]:
df_sold_clean['MLSAreaMajor'].value_counts(normalize=True).head()

MLSAreaMajor
699 - Not Defined                     0.093562
SRCAR - Southwest Riverside County    0.047784
252 - Riverside                       0.013095
248 - Corona                          0.008476
686 - Ontario                         0.007481
Name: proportion, dtype: float64

In [176]:
df_sold_clean['MLSAreaMajor'].describe()

count                521718
unique                 1114
top       699 - Not Defined
freq                  48813
Name: MLSAreaMajor, dtype: object

In [177]:
df_sold_clean['MLSAreaMajor'] = df_sold_clean['MLSAreaMajor'].fillna("Missing")

In [178]:
df_sold_clean['MLSAreaMajor'].value_counts(normalize=True).head()

MLSAreaMajor
Missing                               0.108702
699 - Not Defined                     0.083392
SRCAR - Southwest Riverside County    0.042590
252 - Riverside                       0.011672
248 - Corona                          0.007555
Name: proportion, dtype: float64

In [179]:
df_sold_clean['Levels'].value_counts().head()

Levels
One            307415
Two            177056
ThreeOrMore     25688
MultiSplit      12019
One,Two          1662
Name: count, dtype: int64

In [180]:
df_sold_clean['Levels'].value_counts(normalize=True).head()

Levels
One            0.583353
Two            0.335983
ThreeOrMore    0.048746
MultiSplit     0.022807
One,Two        0.003154
Name: proportion, dtype: float64

In [181]:
df_sold_clean['Levels'] = df_sold_clean['Levels'].fillna("Missing")

In [182]:
df_sold_clean['Levels'].value_counts(normalize=True).head()

Levels
One            0.525185
Two            0.302481
Missing        0.099714
ThreeOrMore    0.043885
MultiSplit     0.020533
Name: proportion, dtype: float64

In [183]:
df_sold_clean['Levels'].isna().sum()

np.int64(0)

Over 50% missing values

In [184]:
over_50 = missing_sold_summary[missing_sold_summary['missing_sold_percent'] >= 50]

In [185]:
over_50

,missing_sold_counts,missing_sold_percent
SubdivisionName,372443,63.627837


In [186]:
df_sold_clean['SubdivisionName'] = df_sold_clean['SubdivisionName'].fillna("Missing")

In [187]:
df_sold_clean['SubdivisionName'].value_counts(normalize=True).head()

SubdivisionName
Missing               0.636278
Other                 0.010197
Not Applicable-1      0.006289
Not Applicable        0.005441
Leisure World (LW)    0.004652
Name: proportion, dtype: float64

In [188]:
df_sold_clean.isna().sum().sort_values(ascending=False)

PurchaseContractDate         12200
BuyerAgentAOR                    0
ListAgentAOR                     0
Flooring                         0
PoolPrivateYN                    0
                             ...  
GarageSpaces_Missing             0
BedroomsTotal_Missing            0
No_Purchase_Date                 0
MainLevelBedrooms_Missing        0
Stories_Missing                  0
Length: 66, dtype: int64

In [189]:
df_sold_clean.isna().sum().sum()

np.int64(12200)

In [190]:
df_sold_clean.isna().sum().sum() / df_sold_clean.notna().sum().sum()

np.float64(0.00031589329600890053)

In [191]:
df_sold_clean.columns

Index(['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'PoolPrivateYN',
       'OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'ListAgentFullName', 'BuyerAgentMlsId', 'BuyerAgentFirstName',
       'BuyerAgentLastName', 'ListingKeyNumeric', 'MLSAreaMajor',
       'CountyOrParish', 'MlsStatus', 'AttachedGarageYN', 'ParkingTotal',
       'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR',
       'YearBuilt', 'StreetNumberNumeric', 'ListingId',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal',
       'ContractStatusChangeDate', 'PurchaseContractDate',
       'ListingContractDate', 'StateOrProvince', 'FireplaceYN', 'Stories',
       'Levels', 'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN',
       'GarageSpaces

#### **Data Consistency Checks**

In [192]:
df_sold_clean['listing_after_close_flag'] = df_sold_clean['ListingContractDate'] > df_sold_clean['CloseDate']

In [193]:
df_sold_clean['listing_after_close_flag'].value_counts()

listing_after_close_flag
False    585238
True        108
Name: count, dtype: int64

In [194]:
df_sold_clean['purchase_after_close_flag'] = df_sold_clean['PurchaseContractDate'] > df_sold_clean['CloseDate']

In [195]:
df_sold_clean['purchase_after_close_flag'].value_counts()

purchase_after_close_flag
False    585222
True        124
Name: count, dtype: int64

In [196]:
df_sold_clean['negative_timeline_flag'] = (
   ( df_sold_clean['ListingContractDate'] > df_sold_clean['PurchaseContractDate']) |
   (df_sold_clean['PurchaseContractDate'] > df_sold_clean['CloseDate']) |
   (df_sold_clean['ListingContractDate'] > df_sold_clean['CloseDate'])
)

In [197]:
df_sold_clean['negative_timeline_flag'].value_counts()

negative_timeline_flag
False    584833
True        513
Name: count, dtype: int64

In [200]:
#df_sold_clean.to_csv("../data/Sold/sold_cleaned.csv", index=False)